<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/main/code/from-scratch/cartpole-dqn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Learning for _cartpole_

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
This work is licensed under the [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---
Based on:
> Udacity, Deep Reinforcement Learning Course. Available on [GitHub](https://github.com/udacity/deep-reinforcement-learning/tree/master/dqn).

## Configuration

In [ ]:
%pip install "gymnasium[classic-control]>=1.1"

In [ ]:
# set render to 'rgb_array' (change to 'human' for local display-enabled runtime)
render_mode = "rgb_array"

In [ ]:
# check whether supporting files exist
import os.path
if not os.path.isfile('dqn_agent.py'):
  !git clone https://github.com/jgromero/drl-csic
  %cd drl-csic/code/from-scratch

## Environment

We will be using [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/), a simple control problem.

<img src="https://github.com/jgromero/drl-csic/blob/main/img/cartpole.gif?raw=true"/>


The action and observation spaces can be printed as follows.

In [ ]:
import gymnasium as gym
env = gym.make("CartPole-v1", render_mode = render_mode)
print(env.observation_space)
print(env.action_space)

We can interact with the environment with an untrained agent by using the `Agent` class in [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/dqn_agent.py).

In [ ]:
from dqn_agent import Agent

agent_untrained = Agent(state_size=4, action_size=2, seed=0)

state, _ = env.reset()

for j in range(500):
    action = agent_untrained.act(state)
    env.render()
    state, reward, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        break

env.close()

## DQN algorithm
Next we provide a generic implementation of Deep Q-Learning (DQN). We rely on external files:

*   [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/dqn_agent.py): Implementation of the agent capabilities (act, step, etc.)
*   [model.py](https://github.com/jgromero/drl-csic/blob/main/code/from-scratch/model.py): Implementation of the neural network of the agent (the Q-network)

The [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) environment is considered solved when the agent obtains an average score of at least $475$ points over 100 consecutive episodes. Episodes are truncated at $500$ steps, which is therefore the maximum score.

In [ ]:
import random
import torch
import numpy as np
from collections import deque
from dqn_agent import device
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
SOLVED_SCORE = 475.0   # average score (last 100 episodes) to consider CartPole-v1 solved

def dqn(agent, n_episodes=2000, max_t=500, eps_start=1.0, eps_end=0.01, eps_decay=0.995, print_every=100):
    """Deep Q-Learning.

    Params
    ======
        agent (Agent): agent to train
        n_episodes (int): maximum number of episodes (n_episodes)
        max_t (int): maximum number of steps per episode (n_steps)
        eps_start (float): epsilon initial value
        eps_end (float): epsilon final value
        eps_decay (float): multiplication factor for epsilon (per episode)
        print_every (int): print score each print_every episodes
    """
    scores = []                        # scores for every episode
    scores_window = deque(maxlen=100)  # scores for the last 100 episodes
    eps = eps_start                    # initialize epsilon

    for i_episode in range(1, n_episodes+1):
        state, _ = env.reset()
        score = 0

        for t in range(max_t):

            # choose action At with e-greedy policy
            action = agent.act(state, eps)

            # apply At and get Rt+1, St+1
            next_state, reward, terminated, truncated, _ = env.step(action)

            # store <St, At, Rt+1, St+1>, train & update
            # (only 'terminated' marks St+1 as terminal; after a truncation we must still bootstrap)
            agent.step(state, action, reward, next_state, terminated)

            # advance to next state
            state = next_state
            score += reward

            if terminated or truncated:
                break

        scores_window.append(score)       # save last score (last 100)
        scores.append(score)              # save last score (every episode)
        eps = max(eps_end, eps_decay*eps) # reduce epsilon

        # print training info
        if i_episode % print_every == 0:
            print('Episode {}\tAverage score (last 100): {:.2f}'.format(i_episode, np.mean(scores_window)))
        if len(scores_window) == 100 and np.mean(scores_window) >= SOLVED_SCORE:
            print('\nProblem solved in {:d} episodes!\tAverage score (last 100): {:.2f}'.format(i_episode-100, np.mean(scores_window)))
            break
    else:
        print('\nProblem not solved in {:d} episodes.\tAverage score (last 100): {:.2f}'.format(n_episodes, np.mean(scores_window)))

    torch.save(agent.qnetwork_local.state_dict(), 'checkpoint-cartpole-dqn.pth') # save weights for trained agent

    return scores

agent = Agent(state_size=4, action_size=2, seed=0)
scores = dqn(agent)

# plot the scores
fig = plt.figure()
ax = fig.add_subplot(111)
plt.plot(np.arange(len(scores)), scores)
plt.ylabel('Score')
plt.xlabel('Episode #')
plt.show()

## Visualize trained agent

We can afterwards load the weights into the agent and visualize how it acts on the environment. (Local run is required for 2D rendering, not Google Colaboratory.)

In [ ]:
# load weights from `checkpoint-cartpole-dqn.pth`
agent_load = Agent(state_size=4, action_size=2, seed=0)

agent_load.qnetwork_local.load_state_dict(torch.load('checkpoint-cartpole-dqn.pth', map_location=device, weights_only=True))

for i in range(3):
    state, _ = env.reset()
    for j in range(500):
        action = agent_load.act(state)
        env.render()
        state, reward, terminated, truncated, _ = env.step(action)
        if terminated or truncated:
            break

env.close()